# **Import libs**

In [1]:
!pip install dask_ml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.0/150.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.2 MB/s eta 0:00:00


In [2]:
import dask.dataframe as dd
import pandas as pd
from dask_ml.model_selection import train_test_split
from sklearn.metrics import classification_report
from dask_ml.linear_model import LogisticRegression

In [3]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# **Load Dataset**

In [4]:
!pip install kaggle

In [5]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d maca11/chess-games-from-lichess-20132014 --unzip

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/maca11/chess-games-from-lichess-20132014
License(s): CC0-1.0
100% 3.32G/3.32G [00:49<00:00, 71.9MB/s]



In [6]:
ddf = dd.read_csv('Lichess_2013_2014_Complete.csv', blocksize='250MB')

## **Prepare**

In [7]:
ddf.partitions[0].compute().columns

Index(['WhiteElo', 'BlackElo', 'WhiteName', 'BlackName', 'Winner',
       'Termination', 'Site', 'Day', 'Month', 'Year', 'InitialTime',
       'Increment', 'TimeControl', 'Opening', 'ECO', 'Number_of_Moves'],
      dtype='object')

In [8]:
ddf = ddf.drop(["Site", 'WhiteName', 'BlackName', 'Opening'], axis=1)

In [9]:
ddf.partitions[0].compute().describe()

,WhiteElo,BlackElo,Day,Month,Year,InitialTime,Increment,Number_of_Moves
count,2142257.00,2142257.00,2142257.00,2142257.00,2142257.00,2142257.00,2142257.00,2142257.00
mean,1610.04,1600.81,15.38,6.37,2013.00,310.55,2.69,33.77
std,211.47,217.83,8.69,2.67,0.01,288.10,4.32,16.21
min,758.00,735.00,1.00,1.00,2012.00,0.00,0.00,1.00
25%,1471.00,1460.00,8.00,4.00,2013.00,120.00,0.00,23.00
50%,1606.00,1595.00,15.00,7.00,2013.00,300.00,0.00,32.00
75%,1748.00,1743.00,23.00,9.00,2013.00,360.00,5.00,43.00
max,2581.00,2584.00,31.00,12.00,2013.00,1800.00,30.00,191.00


In [10]:
ddf.partitions[0].compute().head()

,WhiteElo,BlackElo,Winner,Termination,Day,Month,Year,InitialTime,Increment,TimeControl,ECO,Number_of_Moves
0,1639,1403,White,Normal,31,12,2012,600,8,Rapid,C00,13
1,1654,1919,White,Normal,31,12,2012,480,2,Rapid,D04,18
2,1643,1747,White,Normal,31,12,2012,420,17,Rapid,C50,11
3,1824,1973,Black,Normal,31,12,2012,60,1,Bullet,B12,47
4,1765,1815,Black,Normal,31,12,2012,60,1,Bullet,C00,23


In [11]:
ddf.partitions[0].compute().nunique()

,0
WhiteElo,1693
BlackElo,1701
Winner,3
Termination,3
Day,31
Month,11
Year,2
InitialTime,31
Increment,31
TimeControl,4


In [12]:
ddf.partitions[0].compute().dtypes

,0
WhiteElo,int64
BlackElo,int64
Winner,string[pyarrow]
Termination,string[pyarrow]
Day,int64
Month,int64
Year,int64
InitialTime,int64
Increment,int64
TimeControl,string[pyarrow]


In [13]:
result_map = {'White': -1, 'Draw': 0, 'Black': 1}

ddf['Winner'] = ddf['Winner'].map(result_map, meta=('Winner', 'int64'))

In [14]:
result_map = {'Abandoned': -1, 'Rules infraction': 0, 'Time forfeit': 1, 'Normal': 2}

ddf['Termination'] = ddf['Termination'].map(result_map, meta=('Termination', 'int64'))

In [15]:
result_map = {'Bullet': 0, 'Blitz': 1, 'Rapid': 2, 'Classical': 3}

ddf['TimeControl'] = ddf['TimeControl'].map(result_map, meta=('TimeControl', 'int64'))

In [16]:
ddf['ECO'] = ddf['ECO'].str[0]

ddf = ddf.categorize(columns=['Winner', 'Termination', 'TimeControl', 'ECO'])

ddf = dd.get_dummies(ddf, columns=["ECO"])

eco_cols = [c for c in ddf.columns if c.startswith("ECO_")]
ddf[eco_cols] = ddf[eco_cols].astype("int8")

In [17]:
ddf.partitions[0].compute().head()

,WhiteElo,BlackElo,Winner,Termination,Day,Month,Year,InitialTime,Increment,TimeControl,Number_of_Moves,ECO_A,ECO_B,ECO_C,ECO_D,ECO_E
0,1639,1403,-1,2,31,12,2012,600,8,2,13,0,0,1,0,0
1,1654,1919,-1,2,31,12,2012,480,2,2,18,0,0,0,1,0
2,1643,1747,-1,2,31,12,2012,420,17,2,11,0,0,1,0,0
3,1824,1973,1,2,31,12,2012,60,1,0,47,0,1,0,0,0
4,1765,1815,1,2,31,12,2012,60,1,0,23,0,0,1,0,0


## **Select by criteria**

In [18]:
ddfc = ddf.groupby("Winner", dropna=False, observed=True).agg({"Number_of_Moves": "mean"})

In [19]:
ddfc.head()

,Number_of_Moves
Winner,
-1,32.68
1,33.52
0,55.20


## **Algorithm**

In [20]:
ddfa = ddf.drop(['Winner', 'Termination', 'Increment',
                 'Number_of_Moves', 'ECO_A', 'ECO_B', 'ECO_C', 'ECO_D'], axis=1)

In [21]:
X = ddfa.drop('ECO_E', axis=1)
y = ddfa['ECO_E']

X_arr = X.to_dask_array(lengths=True)
y_arr = y.to_dask_array(lengths=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_arr, y_arr, test_size=0.2, random_state=42, shuffle=True
)

lr = LogisticRegression(solver='lbfgs', max_iter=50, tol=1e-3, random_state=42)
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

/usr/local/lib/python3.12/dist-packages/dask/config.py:835: FutureWarning: Dask configuration key 'fuse_ave_width' has been deprecated; please use 'optimization.fuse.ave-width' instead
  warnings.warn(


In [22]:
y_test_computed, y_pred_computed = dd.compute(y_test, y_pred)
print(classification_report(y_test_computed, y_pred_computed))

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.98      1.00      0.99   2934873
           1       0.00      0.00      0.00     48776

    accuracy                           0.98   2983649
   macro avg       0.49      0.50      0.50   2983649
weighted avg       0.97      0.98      0.98   2983649



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Doesn't bring